# Exercise 1: Basic Deduplication (Record-Level)

## Learning Objectives

In this exercise, you will:
- Load the sample dataset from the project `data/` directory (local to the CAI session)
- Remove exact duplicates with Spark `dropDuplicates()`
- Analyze deduplication results
- Save the cleaned dataset to HDFS under `/tmp`

## Prerequisites

Run **`00_Getting_Started.ipynb`** first to verify local Spark and paths. Default input:

```text
../data/redundant_data.csv
```

## What is Record-Level Deduplication?

**Record-level deduplication** finds and removes duplicate **rows** within a dataset by comparing key columns (here: `name` + `email`).

| Aspect | Record-Level | File-Level |
|--------|--------------|------------|
| **What it finds** | Duplicate rows in a dataset | Duplicate files in storage |
| **Comparison unit** | Individual records | Entire files |
| **Method** | Compare column values | Compare content hashes |

### The `exact` method

- Uses Spark's `dropDuplicates(["name", "email"])`
- Fast and exact — only catches identical key values
- Does **not** catch near-duplicates (e.g. case differences)


## Step 1: Create Local Spark Session


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Exercise1_BasicDeduplication")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark master:  {spark.sparkContext.master}")
print("✓ Spark session created")
print("⚠ Restart kernel if a prior session set fs.defaultFS=hdfs://ns1.")


## Step 2: Load Sample Data (Local)

Default input is the project CSV. To use HDFS instead, set `INPUT_PATH = HDFS_INPUT`.


In [ ]:
from pathlib import Path

LOCAL_INPUT = Path("../data").resolve() / "redundant_data.csv"
HDFS_INPUT = "/tmp/cdp_user_demo/phase1/redundant_data.csv"
HDFS_OUTPUT = "/tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet"

# Default: local project data (file://). Switch to HDFS only if Spark has cluster Hadoop conf.
INPUT_PATH = LOCAL_INPUT.resolve().as_uri()

df = spark.read.csv(INPUT_PATH, header=True, inferSchema=True)

print(f"✓ Loaded: {INPUT_PATH}")
print(f"Total records: {df.count():,}")
print(f"Columns: {', '.join(df.columns)}")
print("\nPreview:")
df.show(10, truncate=False)
df.printSchema()


## Step 3: Inspect Duplicates

Count uniqueness on the key columns before deduplicating.


In [ ]:
KEY_COLS = ["name", "email"]

original_count = df.count()
unique_before = df.select(*KEY_COLS).distinct().count()
duplicates_before = original_count - unique_before

print(f"Total records: {original_count:,}")
print(f"Unique by {KEY_COLS}: {unique_before:,}")
print(f"Duplicate records: {duplicates_before:,}")
print(f"Duplicate rate: {(duplicates_before / original_count * 100) if original_count else 0:.2f}%")


## Step 4: Run Exact Deduplication

Keep one row per unique `name` + `email` combination using Spark `dropDuplicates()`.


In [ ]:
# Exact record-level deduplication on key columns
df_deduped = df.dropDuplicates(KEY_COLS)

unique_count = df_deduped.count()
duplicates_removed = original_count - unique_count
deduplication_rate = (duplicates_removed / original_count * 100) if original_count else 0

print(f"Original records:     {original_count:,}")
print(f"Unique records:       {unique_count:,}")
print(f"Duplicates removed:   {duplicates_removed:,}")
print(f"Deduplication rate:   {deduplication_rate:.2f}%")
print("\nSample of deduplicated rows:")
df_deduped.show(10, truncate=False)


## Step 5: Save Results to HDFS `/tmp`

Write the cleaned dataset as Parquet for reuse in later exercises.


In [ ]:
(
    df_deduped.write
    .mode("overwrite")
    .parquet(HDFS_OUTPUT)
)

print(f"✓ Deduplicated data written to: {HDFS_OUTPUT}")

# Verify
df_out = spark.read.parquet(HDFS_OUTPUT)
print(f"✓ Verified read — rows: {df_out.count():,}")


## Understanding the Results

After running deduplication you should see:
- **Original count**: rows in the input
- **Unique count**: rows remaining after `dropDuplicates`
- **Duplicates removed**: original − unique
- **Deduplication rate**: percent of rows removed
- **Output location**: `hdfs://ns1/tmp/cdp_user_demo/phase1/results/exercise1_exact.parquet`

### Questions to Answer

1. How many duplicates were found?
2. What percentage of records were duplicates?
3. Where are the results saved?
4. Why might records be duplicated in real systems?
5. What happens if two records share a name but have different emails?
   - They are **not** duplicates — keys must match on **all** key columns.

## Key Takeaways

- Record-level deduplication removes duplicate rows by key columns
- Exact matching with `dropDuplicates()` is fast but case/near-match sensitive
- Input defaults to local project `data/`; results are written to HDFS `/tmp`

## Cleanup


In [ ]:
spark.stop()
print("✓ Spark session stopped")
